# Initial and final fidelity distributions

This notebook compares the initial fidelity with the final PyTheus-verified fidelity for GHZ, W, and linear-cluster (LC) inverse-design runs at $n=4,6,8$. It produces one combined $3\times3$ figure, nine individual panels, and a JSON statistics summary.

## Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_repository_root():
    """Support launching Jupyter from the repository root or plotting_code/."""
    current = Path.cwd().resolve()
    if (current / "ml_model.py").is_file() and (current / "plotting_code").is_dir():
        return current
    if current.name == "plotting_code" and (current.parent / "ml_model.py").is_file():
        return current.parent
    raise RuntimeError(
        "Launch this notebook from the PolySurrogate repository root "
        "or from its plotting_code directory."
    )


REPO_ROOT = find_repository_root()
PLOTTING_DIR = REPO_ROOT / "plotting_code"
DATA_DIR = PLOTTING_DIR / "Data" / "runtime_data" / "surrogate_optimisation"
OUTPUT_DIR = PLOTTING_DIR / "Results" / "initial_final_fid"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STATES = ("GHZ", "W", "LC")
NODE_COUNTS = (4, 6, 8)
INPUT_FILES = {
    state: {n: DATA_DIR / state / f"{n}n.json" for n in NODE_COUNTS}
    for state in STATES
}
INITIAL_KEY = "initial_fidelities"
FINAL_KEY = "final_pytheus_fidelities"

VERSION = 2
SAVE_FORMATS = ("png", "pdf", "svg")
SAVE_DPI = 700
HISTOGRAM_BINS = np.linspace(0.0, 1.0, 31)
HISTOGRAM_RWIDTH = 1.0
HISTOGRAM_EDGE_COLOR = "black"
HISTOGRAM_LINEWIDTH = 1.4
HISTOGRAM_ALPHA = 0.75
INITIAL_LABEL = "Initial graphs"
FINAL_LABEL = "Optimised graphs"
X_LIMITS = (-0.05, 1.05)

COMBINED_FIGSIZE = (18, 12)
INDIVIDUAL_FIGSIZE = (6.5, 5.0)
WSPACE = 0.32
HSPACE = 0.42
LEFT_MARGIN = 0.06
RIGHT_MARGIN = 0.99
BOTTOM_MARGIN = 0.07
TOP_MARGIN = 0.98

LABEL_FONTSIZE = 18
TICK_FONTSIZE = 15
LEGEND_FONTSIZE = 13
CORNER_LABEL_FONTSIZE = 18
X_LABELPAD = 6
Y_LABELPAD = 6
TICK_PAD = 4

plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16,
    "lines.linewidth": 2.0,
    "axes.linewidth": 0.9,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "figure.dpi": 150,
    "savefig.dpi": SAVE_DPI,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.unicode_minus": False,
})

all_input_files = [path for files in INPUT_FILES.values() for path in files.values()]
missing_files = [path for path in all_input_files if not path.is_file()]
if missing_files:
    formatted = "\n".join(f"  - {path}" for path in missing_files)
    raise FileNotFoundError(f"Missing fidelity data files:\n{formatted}")

print("Fidelity data:", DATA_DIR)
print("Figure output:", OUTPUT_DIR)
print(f"Validated {len(all_input_files)} input files.")

## Load fidelity data and calculate statistics

In [ ]:
def load_fidelities(path):
    with path.open() as file:
        summary = json.load(file)
    missing_keys = {INITIAL_KEY, FINAL_KEY} - summary.keys()
    if missing_keys:
        raise KeyError(f"{path} is missing fields: {sorted(missing_keys)}")

    initial = np.asarray(summary[INITIAL_KEY], dtype=float).ravel()
    final = np.asarray(summary[FINAL_KEY], dtype=float).ravel()
    if initial.shape != final.shape:
        raise ValueError(
            f"Fidelity shape mismatch in {path}: {initial.shape} != {final.shape}"
        )

    finite = np.isfinite(initial) & np.isfinite(final)
    initial, final = initial[finite], final[finite]
    if initial.size == 0:
        raise ValueError(f"No finite fidelity pairs found in {path}")
    return initial, final


def summarize(state, node_count, path, initial, final):
    return {
        "target": state,
        "n_nodes": node_count,
        "num_samples": initial.size,
        "initial_key": INITIAL_KEY,
        "final_key": FINAL_KEY,
        "initial_mean": initial.mean(),
        "initial_std": initial.std(),
        "initial_min": initial.min(),
        "initial_max": initial.max(),
        "final_mean": final.mean(),
        "final_std": final.std(),
        "final_min": final.min(),
        "final_max": final.max(),
        "summary_path": str(path.relative_to(REPO_ROOT)),
    }


fidelity_data = {state: {} for state in STATES}
statistics = []
for state in STATES:
    for node_count in NODE_COUNTS:
        path = INPUT_FILES[state][node_count]
        initial, final = load_fidelities(path)
        fidelity_data[state][node_count] = {"initial": initial, "final": final}
        statistics.append(summarize(state, node_count, path, initial, final))
        print(
            f"{state}, n={node_count}: {initial.size} samples, "
            f"initial mean={initial.mean():.6f}, final mean={final.mean():.6f}"
        )

## Plotting functions

In [ ]:
CORNER_LABEL_BOX = {
    "boxstyle": "round,pad=0.3",
    "facecolor": "white",
    "edgecolor": "none",
    "alpha": 1.0,
}


def style_axis(ax):
    ax.grid(False)
    ax.tick_params(
        axis="both", labelsize=TICK_FONTSIZE, width=0.8, length=3.5,
        direction="out", pad=TICK_PAD, labelbottom=True, labelleft=True,
    )
    for spine in ax.spines.values():
        spine.set_linewidth(0.9)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")


def draw_panel(ax, state, node_count, initial, final):
    ax.hist(
        initial, bins=HISTOGRAM_BINS, alpha=HISTOGRAM_ALPHA,
        edgecolor=HISTOGRAM_EDGE_COLOR, linewidth=HISTOGRAM_LINEWIDTH,
        rwidth=HISTOGRAM_RWIDTH, label=INITIAL_LABEL,
    )
    ax.hist(
        final, bins=HISTOGRAM_BINS, alpha=HISTOGRAM_ALPHA,
        edgecolor=HISTOGRAM_EDGE_COLOR, linewidth=HISTOGRAM_LINEWIDTH,
        rwidth=HISTOGRAM_RWIDTH, label=FINAL_LABEL,
    )
    ax.set_yscale("log")
    ax.set_xlim(X_LIMITS)
    ax.set_xlabel(
        "Fidelity", fontsize=LABEL_FONTSIZE, fontweight="bold",
        labelpad=X_LABELPAD,
    )
    ax.set_ylabel(
        "Number of samples", fontsize=LABEL_FONTSIZE, fontweight="bold",
        labelpad=Y_LABELPAD,
    )
    style_axis(ax)
    ax.text(
        0.04, 0.94, rf"$\mathrm{{{state}}}_{{{node_count}}}$",
        transform=ax.transAxes, ha="left", va="top",
        fontsize=CORNER_LABEL_FONTSIZE, fontweight="bold",
        bbox=CORNER_LABEL_BOX, zorder=10,
    )
    ax.legend(
        loc="upper center",
        prop={"weight": "bold", "size": LEGEND_FONTSIZE},
        frameon=True, framealpha=0.9, edgecolor="0.75", handlelength=1.2,
        borderpad=0.35, labelspacing=0.3, handletextpad=0.5,
    )


def save_figure(fig, filename):
    for extension in SAVE_FORMATS:
        path = OUTPUT_DIR / f"{filename}.{extension}"
        dpi = SAVE_DPI if extension == "png" else None
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
        print(f"Saved: {path}")
    plt.show()
    plt.close(fig)

## Generate combined and individual figures

In [ ]:
fig, axes = plt.subplots(
    len(STATES), len(NODE_COUNTS), figsize=COMBINED_FIGSIZE,
    sharex=True, sharey=True,
)
for row, state in enumerate(STATES):
    for column, node_count in enumerate(NODE_COUNTS):
        values = fidelity_data[state][node_count]
        draw_panel(
            axes[row, column], state, node_count,
            values["initial"], values["final"],
        )

fig.subplots_adjust(
    left=LEFT_MARGIN, right=RIGHT_MARGIN, bottom=BOTTOM_MARGIN, top=TOP_MARGIN,
    wspace=WSPACE, hspace=HSPACE,
)
save_figure(fig, "initial_final_fidelity_combined")

for state in STATES:
    for node_count in NODE_COUNTS:
        values = fidelity_data[state][node_count]
        fig, ax = plt.subplots(figsize=INDIVIDUAL_FIGSIZE)
        draw_panel(ax, state, node_count, values["initial"], values["final"])
        fig.tight_layout()
        save_figure(fig, f"initial_final_fidelity_{state}_{node_count}")

## Save statistics

In [ ]:
statistics_path = (
    OUTPUT_DIR / f"initial_final_histogram_statistics_3x3_v{VERSION}.json"
)
with statistics_path.open("w") as file:
    json.dump(statistics, file, indent=4)

print("Saved:", statistics_path)